# Entrenar YOLOv8 con dataset RGB auto-etiquetado

Usa las anotaciones en `Data/procesed/predictions_rgb_auto/anotadas` para entrenar un detector YOLO en imagenes visibles (RGB).

---
## Flujo
1. Configuracion
2. Definir clases
3. Convertir XML (Pascal VOC) a formato YOLO (.txt)
4. Dividir en train/val
5. Crear data.yaml
6. Visualizar anotaciones
7. Auto-etiquetar dataset RGB completo
8. Entrenar YOLOv8
9. Evaluar
---

In [ ]:
# ============================================================
# 1. DEPENDENCIAS
# ============================================================
# !pip install ultralytics torch torchvision matplotlib seaborn scikit-learn tqdm

In [42]:
from pathlib import Path
import xml.etree.ElementTree as ET
import random
import shutil
import numpy as np
import matplotlib.pyplot as plt
import yaml
from PIL import Image
from collections import Counter

from ultralytics import YOLO

random.seed(42)
np.random.seed(42)
print("Librerias cargadas")

Librerias cargadas


---
## 2. Configuracion
---

In [43]:
# ============================================================
# 2. CONFIGURACION
# ============================================================

RGB_ANOTADAS_DIR = Path("../Data/procesed/predictions_rgb_auto/anotadas")
RGB_YOLO_DIR = Path("../Data/procesed/yolo_rgb_dataset")
MODELS_DIR = Path("../Models").resolve()
FULL_RGB_DIR = Path("../Data/raw/imagenes_rgb/rgb")
FULL_RGB_OUTPUT = Path("../Data/procesed/predictions_rgb_full")

TRAIN_SPLIT = 0.8
MODEL_SIZE = "n"

MODELS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Dataset RGB: {RGB_ANOTADAS_DIR}")
print(f"Salida YOLO: {RGB_YOLO_DIR}")
print(f"Dataset RGB completo: {FULL_RGB_DIR}")
print(f"Modelos: {MODELS_DIR}")

Dataset RGB: ../Data/procesed/predictions_rgb_auto/anotadas
Salida YOLO: ../Data/procesed/yolo_rgb_dataset
Dataset RGB completo: ../Data/raw/imagenes_rgb/rgb
Modelos: /Users/adamleanos/Projecto ia - detector de billetes/Models


---
## 3. Definir clases
---

In [ ]:
# ============================================================
# 3. CLASES
# ============================================================

CLASSES = [
    "animal_100bs", "animal_10bs", "animal_200bs", "animal_20bs", "animal_50bs",
    "ir_100bs", "ir_10bs", "ir_200bs", "ir_20bs", "ir_50bs",
    "personaje_100bs", "personaje_10bs", "personaje_200bs", "personaje_20bs", "personaje_50bs",
    "serie_a",
    "valor_100bs", "valor_10bs", "valor_200bs", "valor_20bs", "valor_50bs",
]
NUM_CLASSES = len(CLASSES)
CLS_TO_ID = {c: i for i, c in enumerate(CLASSES)}

print(f"Total clases: {NUM_CLASSES}")
for i, c in enumerate(CLASSES):
    print(f"  {i}: {c}")

---
## 4. Convertir XML a YOLO y preparar dataset
---

In [ ]:
# ============================================================
# 4. CONVERTIR VOC -> YOLO y DIVIDIR
# ============================================================

def voc_to_yolo(xml_path, cls_to_id):
    tree = ET.parse(xml_path)
    root = tree.getroot()
    w = int(root.find(".//size/width").text)
    h = int(root.find(".//size/height").text)
    labels = []
    for obj in root.findall(".//object"):
        name = obj.find("name").text
        if name not in cls_to_id:
            continue
        cls_id = cls_to_id[name]
        xmin = int(obj.find(".//bndbox/xmin").text)
        ymin = int(obj.find(".//bndbox/ymin").text)
        xmax = int(obj.find(".//bndbox/xmax").text)
        ymax = int(obj.find(".//bndbox/ymax").text)
        cx = (xmin + xmax) / 2 / w
        cy = (ymin + ymax) / 2 / h
        bw = (xmax - xmin) / w
        bh = (ymax - ymin) / h
        labels.append(f"{cls_id} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")
    return labels

def preparar_dataset_yolo(source_dir, destino, cls_to_id, split=0.8):
    if destino.exists():
        print(f"{destino} ya existe. Eliminalo para regenerar.")
        return

    images_train = destino / "images/train"
    images_val = destino / "images/val"
    labels_train = destino / "labels/train"
    labels_val = destino / "labels/val"
    for d in [images_train, images_val, labels_train, labels_val]:
        d.mkdir(parents=True, exist_ok=True)

    todas = []
    for img_path in sorted(source_dir.rglob("*.png")):
        xml_path = img_path.with_suffix(".xml")
        if xml_path.exists():
            todas.append((img_path, xml_path))

    random.shuffle(todas)
    n_train = int(split * len(todas))

    for i, (img_path, xml_path) in enumerate(todas):
        prefix = "train" if i < n_train else "val"
        shutil.copy2(str(img_path), str(destino / f"images/{prefix}/{img_path.name}"))
        labels = voc_to_yolo(xml_path, cls_to_id)
        if labels:
            label_path = destino / f"labels/{prefix}/{img_path.stem}.txt"
            label_path.write_text("\n".join(labels))

    print(f"Dataset listo en {destino}")
    print(f"  Train: {n_train} imagenes")
    print(f"  Val: {len(todas) - n_train} imagenes")

preparar_dataset_yolo(RGB_ANOTADAS_DIR, RGB_YOLO_DIR, CLS_TO_ID, TRAIN_SPLIT)

---
## 5. Crear data.yaml
---

In [ ]:
# ============================================================
# 5. DATA.YAML
# ============================================================

rgb_data_yaml = {
    "path": str(RGB_YOLO_DIR.resolve()),
    "train": "images/train",
    "val": "images/val",
    "nc": NUM_CLASSES,
    "names": CLASSES,
}

rgb_yaml_path = RGB_YOLO_DIR / "data.yaml"
with open(rgb_yaml_path, "w") as f:
    yaml.dump(rgb_data_yaml, f, default_flow_style=False)

print(f"data.yaml creado en {rgb_yaml_path}")
print(yaml.dump(rgb_data_yaml, default_flow_style=False))

---
## 6. Visualizar anotaciones
---

In [ ]:
# ============================================================
# 6. VISUALIZAR ANOTACIONES
# ============================================================

def show_annotations(split="train", num=5):
    img_dir = RGB_YOLO_DIR / f"images/{split}"
    label_dir = RGB_YOLO_DIR / f"labels/{split}"
    imgs = sorted(img_dir.glob("*.png"))
    if not imgs:
        print(f"No hay imagenes en {img_dir}")
        return
    samples = random.sample(imgs, min(num, len(imgs)))

    fig, axes = plt.subplots(1, len(samples), figsize=(4 * len(samples), 4))
    if len(samples) == 1:
        axes = [axes]

    colors = [
        "red", "blue", "green", "orange", "purple", "cyan",
        "magenta", "brown", "pink", "olive", "navy",
    ]

    for ax, img_path in zip(axes, samples):
        img = Image.open(img_path).convert("RGB")
        ax.imshow(img)
        h, w = img.size[1], img.size[0]

        label_file = label_dir / f"{img_path.stem}.txt"
        if label_file.exists():
            for line in label_file.read_text().strip().split("\n"):
                if not line:
                    continue
                parts = line.strip().split()
                cls_id, cx, cy, bw, bh = map(float, parts[:5])
                x1 = int((cx - bw / 2) * w)
                y1 = int((cy - bh / 2) * h)
                bx = int(bw * w)
                by = int(bh * h)
                c = colors[int(cls_id) % len(colors)]
                rect = plt.Rectangle(
                    (x1, y1), bx, by, fill=False, edgecolor=c, linewidth=2
                )
                ax.add_patch(rect)
                ax.text(
                    x1, y1 - 2, CLASSES[int(cls_id)],
                    color=c, fontsize=7, fontweight="bold",
                )
        ax.axis("off")

    plt.suptitle(f"Anotaciones RGB - {split}")
    plt.tight_layout()
    plt.show()


show_annotations("train")
show_annotations("val")

---
## 7. Auto-etiquetar dataset RGB completo
---

In [44]:
# ============================================================
# 7. AUTO-ETIQUETAR DATASET RGB COMPLETO
# ============================================================

def auto_etiquetar(model, source_dir, output_dir, conf=0.85):
    source_dir = Path(source_dir)
    output_dir = Path(output_dir)

    imagenes = sorted(source_dir.rglob("*.png"))
    print(f"Procesando {len(imagenes)} imagenes de {source_dir.name}...")

    stats = Counter()
    total_det = 0
    imgs_con_det = 0

    for img_path in imagenes:
        img_pil = Image.open(img_path)
        w, h = img_pil.size
        rel = img_path.relative_to(source_dir)

        out_path_img = output_dir / rel
        out_path_img.parent.mkdir(parents=True, exist_ok=True)
        out_path_xml = out_path_img.with_suffix(".xml")

        results = model(str(img_path), conf=conf, verbose=False)
        boxes = results[0].boxes

        if boxes is not None and len(boxes) > 0:
            imgs_con_det += 1
            total_det += len(boxes)

            annotation = ET.Element("annotation")
            folder = ET.SubElement(annotation, "folder")
            folder.text = str(rel.parent)
            filename = ET.SubElement(annotation, "filename")
            filename.text = img_path.name
            path_elem = ET.SubElement(annotation, "path")
            path_elem.text = str(out_path_img)
            source = ET.SubElement(annotation, "source")
            database = ET.SubElement(source, "database")
            database.text = "YOLOv8 Auto-label"
            size = ET.SubElement(annotation, "size")
            width = ET.SubElement(size, "width")
            width.text = str(w)
            height = ET.SubElement(size, "height")
            height.text = str(h)
            depth = ET.SubElement(size, "depth")
            depth.text = "3"
            segmented = ET.SubElement(annotation, "segmented")
            segmented.text = "0"

            for box in boxes:
                cls_id = int(box.cls[0])
                stats[CLASSES[cls_id]] += 1
                x1, y1, x2, y2 = map(int, box.xyxy[0])

                obj = ET.SubElement(annotation, "object")
                name = ET.SubElement(obj, "name")
                name.text = CLASSES[cls_id]
                pose = ET.SubElement(obj, "pose")
                pose.text = "Unspecified"
                truncated = ET.SubElement(obj, "truncated")
                truncated.text = "0"
                difficult = ET.SubElement(obj, "difficult")
                difficult.text = "0"
                bndbox = ET.SubElement(obj, "bndbox")
                xmin = ET.SubElement(bndbox, "xmin")
                xmin.text = str(x1)
                ymin = ET.SubElement(bndbox, "ymin")
                ymin.text = str(y1)
                xmax = ET.SubElement(bndbox, "xmax")
                xmax.text = str(x2)
                ymax = ET.SubElement(bndbox, "ymax")
                ymax.text = str(y2)

            def indent_xml(elem, level=0):
                i = "\n" + level * "\t"
                if len(elem):
                    if not elem.text or not elem.text.strip():
                        elem.text = i + "\t"
                    if not elem.tail or not elem.tail.strip():
                        elem.tail = i
                    for child in elem:
                        indent_xml(child, level + 1)
                    if not child.tail or not child.tail.strip():
                        child.tail = i
                else:
                    if level and (not elem.tail or not elem.tail.strip()):
                        elem.tail = i

            indent_xml(annotation)
            tree = ET.ElementTree(annotation)
            tree.write(out_path_xml, encoding="utf-8", xml_declaration=True)

        img_pil.save(str(out_path_img))

    print(f"\nProcesado completado")
    print(f"  Imagenes con detecciones: {imgs_con_det}/{len(imagenes)}")
    print(f"  Total detecciones: {total_det}")
    print(f"\nDetecciones por clase:")
    for cls_name, count in stats.most_common():
        print(f"  {cls_name}: {count}")
    print(f"\nXMLs e imagenes guardados en: {output_dir}")


best_model_path = MODELS_DIR / "yolo_rgb_detector/weights/best.pt"
model_full = YOLO(str(best_model_path))

auto_etiquetar(model_full, FULL_RGB_DIR, FULL_RGB_OUTPUT / "anotadas", conf=0.55)

Procesando 2126 imagenes de rgb...

Procesado completado
  Imagenes con detecciones: 2125/2126
  Total detecciones: 4098

Detecciones por clase:
  serie_a: 1060
  valor_200bs: 252
  valor_100bs: 246
  valor_20bs: 244
  valor_50bs: 244
  valor_10bs: 240
  ir_10bs: 185
  ir_50bs: 181
  ir_100bs: 180
  ir_200bs: 180
  ir_20bs: 180
  personaje_10bs: 95
  personaje_50bs: 91
  personaje_100bs: 90
  personaje_200bs: 90
  personaje_20bs: 90
  animal_100bs: 90
  animal_10bs: 90
  animal_200bs: 90
  animal_20bs: 90
  animal_50bs: 90

XMLs e imagenes guardados en: ../Data/procesed/predictions_rgb_full/anotadas


---
## 8. Entrenar YOLOv8
---

In [ ]:
# ============================================================
# 8. ENTRENAR
# ============================================================
import os
os.environ["PYTORCH_MPS_HIGH_WATERMARK_RATIO"] = "0.0"

model = YOLO(f"yolov8{MODEL_SIZE}.pt")

results = model.train(
    data=str(rgb_yaml_path),
    epochs=100,
    imgsz=320,
    batch=32,
    device="mps",
    patience=15,
    save=True,
    project=str(MODELS_DIR),
    name="yolo_rgb_detector",
    exist_ok=True,
    plots=True,
    verbose=True,
    optimizer='auto',
)
print("\nEntrenamiento completado")

---
## 9. Evaluar en validacion
---

In [ ]:
# ============================================================
# 9. EVALUACION
# ============================================================

results_dir = MODELS_DIR / "yolo_rgb_detector"
if not results_dir.exists():
    results_dir = Path("runs/Models/yolo_rgb_detector")

best_model_path = results_dir / "weights/best.pt"
print(f"Modelo: {best_model_path}")

if best_model_path.exists():
    model_eval = YOLO(str(best_model_path))
    metrics = model_eval.val(data=str(rgb_yaml_path))
    print("\nMetricas:")
    print(f"  mAP50: {metrics.box.map50:.4f}")
    print(f"  mAP50-95: {metrics.box.map:.4f}")
    print(f"  Precision: {metrics.box.mp:.4f}")
    print(f"  Recall: {metrics.box.mr:.4f}")
else:
    print("No hay modelo entrenado aun.")